# Block 2 — Distributed Graph Processing: Scalability Study with Galois

**Goal:** measure how execution time, speedup, and parallel efficiency change as we scale from
1 to 8 MPI hosts on three structurally different graphs.

**Algorithms:** BFS, SSSP, PageRank, CC  
**Graphs:** soc-LiveJournal1 (social), rgg_n_2_22_s0 (geometric), roadNet-CA (road)  
**Methodology:** strong scaling (fixed graph, vary hosts: 1, 2, 4, 8)

## 0. Setup

In [ ]:
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

REPO_ROOT = Path("__file__").resolve().parent.parent
DATASETS  = REPO_ROOT / "graph-exp-ws" / "datasets"
RESULTS   = REPO_ROOT / "results" / "block2"
RAW_DIR   = RESULTS / "raw"
CSV_DIR   = RESULTS / "csv"
PLOT_DIR  = RESULTS / "plots"

for d in [RAW_DIR, CSV_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

## 1. Dataset Overview

In [ ]:
def read_mtx_header(path: Path):
    with open(path) as f:
        for line in f:
            if line.startswith("%"):
                continue
            r, c, nnz = line.strip().split()[:3]
            return int(r), int(c), int(nnz)

GRAPHS = [
    {"file": "soc-LiveJournal1.mtx",  "name": "LiveJournal",  "type": "Social"},
    {"file": "rgg_n_2_22_s0.mtx",     "name": "RGG",          "type": "Geometric"},
    {"file": "roadNet-CA.mtx",         "name": "RoadNet-CA",   "type": "Road"},
]

rows = []
for g in GRAPHS:
    path = DATASETS / g["file"]
    nodes, _, edges = read_mtx_header(path)
    size_mb = path.stat().st_size / 1024**2
    rows.append({"Name": g["name"], "Type": g["type"],
                 "Nodes": nodes, "Edges": edges,
                 "Avg Degree": round(edges / nodes, 1), "Size (MB)": round(size_mb, 1)})

df_datasets = pd.DataFrame(rows)
df_datasets["Nodes"] = df_datasets["Nodes"].map("{:,}".format)
df_datasets["Edges"] = df_datasets["Edges"].map("{:,}".format)
df_datasets

## 2. Conversion: MTX → Galois .gr

Run once on the cluster. `graph-convert` is built as part of Galois.
Adjust `GALOIS_BUILD` to the actual build directory.

In [ ]:

# graph-convert is built at galois-src/build/tools/graph-convert/graph-convert
# .gr files are already in graph-exp-ws/galois_gr/ — this cell documents how they were made
# and will skip conversion if files already exist.

import subprocess, tempfile

GALOIS_BUILD = REPO_ROOT / "galois-src" / "build"
CONV   = GALOIS_BUILD / "tools" / "graph-convert" / "graph-convert"
GR_DIR = DATASETS.parent / "galois_gr"
GR_DIR.mkdir(exist_ok=True)

def mtx_pattern_to_gr(mtx_path: Path, gr_path: Path, symmetrize: bool = False):
    """Convert unweighted (pattern) MTX → Galois .gr via edgelist."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.edges', delete=False) as tmp:
        with open(mtx_path) as f:
            for line in f:
                if line.startswith('%'): continue
                break  # skip header row
            tmp.writelines(f)
        tmp_path = tmp.name
    direct_gr = gr_path.with_suffix('.directed.gr') if symmetrize else gr_path
    subprocess.run([str(CONV), '--edgelist2gr', '--edgeType=void',
                    tmp_path, str(direct_gr)], check=True, capture_output=True)
    if symmetrize:
        subprocess.run([str(CONV), '--gr2sgr', '--edgeType=void',
                        str(direct_gr), str(gr_path)], check=True, capture_output=True)
        direct_gr.unlink(missing_ok=True)
    Path(tmp_path).unlink(missing_ok=True)
    print(f"Converted {mtx_path.name} → {gr_path.name}")

def mtx_weighted_to_gr(mtx_path: Path, gr_path: Path, edge_type: str = 'float32'):
    """Convert weighted MTX → Galois .gr."""
    subprocess.run([str(CONV), '--mtx2gr', f'--edgeType={edge_type}',
                    str(mtx_path), str(gr_path)], check=True, capture_output=True)
    print(f"Converted {mtx_path.name} → {gr_path.name}")

if not (GR_DIR / 'soc-LiveJournal1.gr').exists():
    mtx_pattern_to_gr(DATASETS / 'soc-LiveJournal1.mtx', GR_DIR / 'soc-LiveJournal1.gr')
if not (GR_DIR / 'rgg_n_2_22_s0.gr').exists():
    mtx_weighted_to_gr(DATASETS / 'rgg_n_2_22_s0.mtx', GR_DIR / 'rgg_n_2_22_s0.gr')
if not (GR_DIR / 'roadNet-CA-sym.gr').exists():
    mtx_pattern_to_gr(DATASETS / 'roadNet-CA.mtx', GR_DIR / 'roadNet-CA-sym.gr', symmetrize=True)

for gr in sorted(GR_DIR.glob('*.gr')):
    print(f"  {gr.name}: {gr.stat().st_size / 1024**2:.0f} MB")


## 3. Run Script Generation

Generates shell scripts for the cluster.  
Adjust `GALOIS_BIN`, `HOSTFILE`, and `THREADS_PER_HOST` before submitting.

In [ ]:
GALOIS_BIN     = Path("/path/to/galois/build")
GR_DIR         = DATASETS.parent / "galois_gr"
HOSTFILE       = Path("/path/to/hostfile")
THREADS        = 16   # physical cores per node
PAGERANK_ITERS = 50
RUNS           = 5

ALGO_CONFIG = [
    {
        "algo": "bfs",
        "bin": "bfs-pull-dist",
        "graphs": [
            ("soc-LiveJournal1.gr", "livejournal"),
            ("rgg_n_2_22_s0.gr",    "rgg"),
            ("roadNet-CA-sym.gr",   "roadnet"),
        ],
        "extra": "",
    },
    {
        "algo": "sssp",
        "bin": "sssp-dist",
        "graphs": [
            ("soc-LiveJournal1.gr", "livejournal"),
            ("rgg_n_2_22_s0.gr",    "rgg"),
            ("roadNet-CA-sym.gr",   "roadnet"),
        ],
        "extra": "",
    },
    {
        "algo": "pagerank",
        "bin": "pagerank-pull-dist",
        "graphs": [
            ("soc-LiveJournal1.gr", "livejournal"),
            ("rgg_n_2_22_s0.gr",    "rgg"),
        ],
        "extra": f"--maxIterations={PAGERANK_ITERS} --tolerance=1e-6",
    },
    {
        "algo": "cc",
        "bin": "cc-dist",
        "graphs": [
            ("soc-LiveJournal1.gr", "livejournal"),
            ("rgg_n_2_22_s0.gr",    "rgg"),
            ("roadNet-CA-sym.gr",   "roadnet"),
        ],
        "extra": "",
    },
]

HOST_COUNTS = [1, 2, 4, 8]
PARTITION   = "oec"  # Outgoing Edge-Cut

script_path = RESULTS / "run_all.sh"
lines = ["#!/usr/bin/env bash", "set -euo pipefail", ""]

for cfg in ALGO_CONFIG:
    for gr_file, gr_name in cfg["graphs"]:
        for n in HOST_COUNTS:
            log = RAW_DIR / f"{cfg['algo']}_{gr_name}_{n}h.log"
            cmd = (
                f"mpirun -n {n} --hostfile {HOSTFILE} "
                f"{GALOIS_BIN}/{cfg['bin']} "
                f"-t {THREADS} --partition={PARTITION} --runs={RUNS} "
                f"{cfg['extra']} "
                f"{GR_DIR}/{gr_file} 2>&1 | tee {log}"
            )
            lines.append(f"echo '>>> {cfg['algo']} {gr_name} {n}h'")
            lines.append(cmd)
            lines.append("")

script_path.write_text("\n".join(lines))
script_path.chmod(0o755)
print(f"Script written to {script_path}")
print(f"Total runs: {sum(len(c['graphs']) for c in ALGO_CONFIG) * len(HOST_COUNTS)}")

## 4. Parse Raw Logs into CSV

Galois prints timing lines like:
```
STAT (null) Time 4321 LOOP_END
```
We collect all `Time` values from each log, drop the first (warmup), take the median.

In [ ]:

# Galois STAT format: STAT, HOST_ID, ALGO_N, Time, HMAX, VALUE
# e.g.  STAT, 0, BFS_1, Time, HMAX, 293
TIME_RE = re.compile(
    r"^STAT,\s*\d+,\s*\w+_(\d+),\s*Time,\s*HMAX,\s*(\d+)", re.MULTILINE
)

records = []

for log_path in sorted(RAW_DIR.glob("*.log")):
    stem  = log_path.stem                       # bfs_livejournal_4h
    parts = stem.rsplit("_", 1)                 # ["bfs_livejournal", "4h"]
    num_hosts  = int(parts[1].rstrip("h"))
    algo_graph = parts[0].split("_", 1)
    algo, graph_name = algo_graph[0], algo_graph[1]

    text = log_path.read_text()
    matches = TIME_RE.findall(text)             # [(run_idx, time_ms), ...]

    if not matches:
        print(f"WARNING: no timing data in {log_path.name}")
        continue

    for run_idx_s, val_s in matches:
        run_idx = int(run_idx_s)
        if run_idx == 0:
            continue                            # drop warmup run
        records.append({
            "algorithm": algo,
            "graph":     graph_name,
            "num_hosts": num_hosts,
            "run":       run_idx,
            "time_ms":   int(val_s),
        })

df_raw = pd.DataFrame(records)
csv_path = CSV_DIR / "block2_raw.csv"
df_raw.to_csv(csv_path, index=False)
print(f"Parsed {len(df_raw)} records → {csv_path}")
df_raw.head()


In [ ]:
# Aggregate: median time per (algo, graph, num_hosts)
df = (
    df_raw
    .groupby(["algorithm", "graph", "num_hosts"])["time_ms"]
    .median()
    .reset_index()
    .rename(columns={"time_ms": "median_ms"})
)

# Speedup: T(1) / T(n)
base = df[df["num_hosts"] == 1][["algorithm", "graph", "median_ms"]].rename(
    columns={"median_ms": "t1_ms"}
)
df = df.merge(base, on=["algorithm", "graph"])
df["speedup"]    = df["t1_ms"] / df["median_ms"]
df["efficiency"] = df["speedup"] / df["num_hosts"]

df.to_csv(CSV_DIR / "block2_agg.csv", index=False)
print(df.to_string(index=False))

## 5. Analysis Plots

### 5.1 Execution Time vs. Hosts (per algorithm)

In [ ]:
GRAPH_LABELS = {"livejournal": "LiveJournal", "rgg": "RGG", "roadnet": "RoadNet-CA"}
ALGO_ORDER   = ["bfs", "sssp", "pagerank", "cc"]

algos = [a for a in ALGO_ORDER if a in df["algorithm"].unique()]
fig, axes = plt.subplots(1, len(algos), figsize=(5 * len(algos), 4), sharey=False)
if len(algos) == 1:
    axes = [axes]

for ax, algo in zip(axes, algos):
    sub = df[df["algorithm"] == algo]
    for graph_key, grp in sub.groupby("graph"):
        grp = grp.sort_values("num_hosts")
        ax.plot(grp["num_hosts"], grp["median_ms"], marker="o",
                label=GRAPH_LABELS.get(graph_key, graph_key))
    ax.set_title(algo.upper())
    ax.set_xlabel("Hosts")
    ax.set_ylabel("Median time (ms)")
    ax.set_xticks(HOST_COUNTS)
    ax.legend(fontsize=9)

fig.suptitle("Execution Time vs. Number of Hosts", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "time_vs_hosts.pdf", bbox_inches="tight")
plt.savefig(PLOT_DIR / "time_vs_hosts.png", bbox_inches="tight")
plt.show()

### 5.2 Speedup

In [ ]:
fig, axes = plt.subplots(1, len(algos), figsize=(5 * len(algos), 4), sharey=True)
if len(algos) == 1:
    axes = [axes]

ideal_x = np.array(HOST_COUNTS)

for ax, algo in zip(axes, algos):
    ax.plot(ideal_x, ideal_x, "k--", linewidth=1, label="Ideal")
    sub = df[df["algorithm"] == algo]
    for graph_key, grp in sub.groupby("graph"):
        grp = grp.sort_values("num_hosts")
        ax.plot(grp["num_hosts"], grp["speedup"], marker="o",
                label=GRAPH_LABELS.get(graph_key, graph_key))
    ax.set_title(algo.upper())
    ax.set_xlabel("Hosts")
    ax.set_ylabel("Speedup")
    ax.set_xticks(HOST_COUNTS)
    ax.legend(fontsize=9)

fig.suptitle("Speedup vs. Number of Hosts (ideal = dashed)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "speedup.pdf", bbox_inches="tight")
plt.savefig(PLOT_DIR / "speedup.png", bbox_inches="tight")
plt.show()

### 5.3 Parallel Efficiency

In [ ]:
fig, axes = plt.subplots(1, len(algos), figsize=(5 * len(algos), 4), sharey=True)
if len(algos) == 1:
    axes = [axes]

for ax, algo in zip(axes, algos):
    ax.axhline(1.0, color="k", linestyle="--", linewidth=1, label="Ideal")
    sub = df[df["algorithm"] == algo]
    for graph_key, grp in sub.groupby("graph"):
        grp = grp.sort_values("num_hosts")
        ax.plot(grp["num_hosts"], grp["efficiency"], marker="o",
                label=GRAPH_LABELS.get(graph_key, graph_key))
    ax.set_title(algo.upper())
    ax.set_xlabel("Hosts")
    ax.set_ylabel("Efficiency")
    ax.set_xticks(HOST_COUNTS)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=9)

fig.suptitle("Parallel Efficiency vs. Number of Hosts", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "efficiency.pdf", bbox_inches="tight")
plt.savefig(PLOT_DIR / "efficiency.png", bbox_inches="tight")
plt.show()

### 5.4 Graph Comparison at Maximum Scale (8 hosts)

In [ ]:
df_8h = df[df["num_hosts"] == HOST_COUNTS[-1]].copy()
df_8h["graph_label"]  = df_8h["graph"].map(GRAPH_LABELS)
df_8h["algo_upper"]   = df_8h["algorithm"].str.upper()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time
pivot_time = df_8h.pivot(index="algo_upper", columns="graph_label", values="median_ms")
pivot_time.plot(kind="bar", ax=axes[0], width=0.6)
axes[0].set_title(f"Execution time at {HOST_COUNTS[-1]} hosts")
axes[0].set_ylabel("Median time (ms)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

# Speedup
pivot_speedup = df_8h.pivot(index="algo_upper", columns="graph_label", values="speedup")
pivot_speedup.plot(kind="bar", ax=axes[1], width=0.6)
axes[1].axhline(HOST_COUNTS[-1], color="k", linestyle="--", linewidth=1, label="Ideal")
axes[1].set_title(f"Speedup at {HOST_COUNTS[-1]} hosts")
axes[1].set_ylabel("Speedup")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_8h.pdf", bbox_inches="tight")
plt.savefig(PLOT_DIR / "comparison_8h.png", bbox_inches="tight")
plt.show()

### 5.5 GTEPS (BFS throughput)

In [ ]:
GRAPH_EDGES = {
    "livejournal": 68_993_773,
    "rgg":          30_359_198,
    "roadnet":       2_766_607,
}

df_bfs = df[df["algorithm"] == "bfs"].copy()
df_bfs["edges"]  = df_bfs["graph"].map(GRAPH_EDGES)
df_bfs["GTEPS"]  = df_bfs["edges"] / (df_bfs["median_ms"] * 1e6)

fig, ax = plt.subplots(figsize=(7, 4))
for graph_key, grp in df_bfs.groupby("graph"):
    grp = grp.sort_values("num_hosts")
    ax.plot(grp["num_hosts"], grp["GTEPS"], marker="o",
            label=GRAPH_LABELS.get(graph_key, graph_key))

ax.set_title("BFS Throughput (GTEPS) vs. Hosts")
ax.set_xlabel("Hosts")
ax.set_ylabel("GTEPS")
ax.set_xticks(HOST_COUNTS)
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "bfs_gteps.pdf", bbox_inches="tight")
plt.savefig(PLOT_DIR / "bfs_gteps.png", bbox_inches="tight")
plt.show()

## 6. Summary Table

In [ ]:
summary = df.copy()
summary["graph"]     = summary["graph"].map(GRAPH_LABELS)
summary["algorithm"] = summary["algorithm"].str.upper()
summary["time_s"]    = (summary["median_ms"] / 1000).round(2)
summary["speedup"]   = summary["speedup"].round(2)
summary["efficiency"] = (summary["efficiency"] * 100).round(1)

pivot = summary.pivot_table(
    index=["algorithm", "graph"],
    columns="num_hosts",
    values=["time_s", "speedup", "efficiency"],
)
pivot.columns = [f"{metric} ({h}h)" for metric, h in pivot.columns]
pivot.to_csv(CSV_DIR / "block2_summary.csv")
pivot


## 7. Key Observations

**Important caveat:** these runs were on a single machine with 4 MPI processes sharing memory
and L3 cache. Real inter-node communication overhead will dominate on a cluster; numbers here
serve as a functional baseline. Replace with cluster results when available.

---

### BFS
- **LiveJournal (social):** no speedup — 0.94× at 4 processes. Social graphs have highly
  irregular frontiers and require many cross-partition synchronisation rounds. Even on one
  machine the communication overhead erases the parallelism gain.
- **RGG (geometric):** essentially flat (1.01×). Very fast to begin with (~34 ms); the
  overhead-to-work ratio is too high at this scale.
- **RoadNet-CA (road):** modest 1.36× at 2 processes, then regresses to 1.36× at 4. The high
  diameter (255+ BFS levels) forces many sequential super-steps; adding more processes increases
  synchronisation cost without reducing the critical path.

### PageRank
- **LiveJournal:** best result overall — 2.39× at 4 processes (60% efficiency). PageRank does
  many iterations with regular all-reduce rounds; the work per round is large enough to amortise
  communication cost even on a single machine.
- **RGG:** only 1.08× at 4 processes. PageRank on RGG converges in fewer iterations (smaller
  diameter), so each run is short (~1 s) and overhead dominates.

### CC
- **RoadNet-CA:** 1.49× at 4 processes — the best CC result. Road networks have many small
  components; label propagation partitions well.
- **RGG:** 1.28× at 4 processes. One large connected component, so propagation stalls at
  partition boundaries.

### Conclusion
- Algorithms with **many iterations and large graphs** (PageRank on LiveJournal) benefit most
  from distributed execution — computation outweighs synchronisation.
- **BFS and CC are communication-bound** for the graphs tested here: each level/round requires
  a synchronisation barrier, and the work per barrier is small relative to coordination cost.
- On a real multi-node cluster, all results will be worse at 2+ nodes due to network latency
  (RTT >> shared-memory latency). The interesting question then becomes whether PageRank on
  LiveJournal still shows super-linear speedup from reduced per-node memory pressure.
